# Aneurysm Volume Prediction v5 (From v2_attention_plus)

这是基于你当前最佳 `v2_attention_plus(0.7976)` 的稳健增强版。

改进点：
1. 训练损失：`Focal Tversky + BCE`（针对小目标/类别不平衡）
2. 训练稳定化：`EMA` 权重用于验证与推理
3. 推理后处理：3D 连通域过滤（largest component / min-size / confidence）
4. OOF自动调参：阈值 + 软硬融合 + 后处理参数 + 线性校准
5. 继续使用 attention + 多seed + TTA + 分层CV


In [ ]:
# 如缺包请取消注释
# %pip install nibabel scipy scikit-learn tqdm pandas matplotlib
# %pip install torch torchvision torchaudio


In [ ]:
import os
import sys
import json
import random
import time
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from scipy import ndimage

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
torch.backends.cudnn.benchmark = True

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = (DEVICE.type == "cuda")

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("Device:", DEVICE, "AMP:", USE_AMP)


In [ ]:
# ===== 路径适配（本地 / 平台 / Kaggle）=====
DATA_ROOT = os.environ.get("ANEURYSM_DATA_ROOT", "").strip()

def is_valid_dataset_dir(p: Path) -> bool:
    return (p / "train").exists() and (p / "test").exists() and (p / "train_labels").exists()

def find_base_dir():
    if DATA_ROOT:
        p = Path(DATA_ROOT)
        if is_valid_dataset_dir(p):
            return p
    candidates = [
        "/dataset/public",
        "/dataset",
        "/kaggle/input/aneurysm-volume-prediction",
        "/kaggle/input/aneurysm-volume",
        "/Users/songling/Desktop/Aneurysm Volume Prediction",
    ]
    for c in candidates:
        p = Path(c)
        if is_valid_dataset_dir(p):
            return p
    for root in [Path("/dataset"), Path("/kaggle/input"), Path.cwd()]:
        if not root.exists():
            continue
        for d in root.rglob("*"):
            if d.is_dir() and is_valid_dataset_dir(d):
                return d
    return None

BASE_DIR = find_base_dir()
if BASE_DIR is None:
    raise FileNotFoundError("未找到数据目录，请设置 ANEURYSM_DATA_ROOT")

TRAIN_IMG_DIR = BASE_DIR / "train"
TRAIN_MASK_DIR = BASE_DIR / "train_labels"
TEST_IMG_DIR = BASE_DIR / "test"
TRAIN_CSV = BASE_DIR / "train.csv"

if Path("/root/setup/solution/working").exists():
    OUTPUT_ROOT = Path("/root/setup/solution/working")
elif Path("/working").exists():
    OUTPUT_ROOT = Path("/working")
elif Path("/kaggle/working").exists():
    OUTPUT_ROOT = Path("/kaggle/working")
else:
    OUTPUT_ROOT = BASE_DIR / "working"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = os.environ.get("RUN_NAME", "").strip() or datetime.now().strftime("v5_ema_ftl_%Y%m%d_%H%M%S")
EXP_DIR = OUTPUT_ROOT / RUN_NAME
CKPT_DIR = EXP_DIR / "checkpoints"
PRED_DIR = EXP_DIR / "predictions"
LOG_DIR = EXP_DIR / "logs"
for d in [EXP_DIR, CKPT_DIR, PRED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("EXP_DIR:", EXP_DIR)


In [ ]:
def load_nii(path):
    nii = nib.load(str(path))
    arr = nii.get_fdata(dtype=np.float32)
    spacing = nii.header.get_zooms()[:3]
    return arr, spacing

def robust_zscore(x, eps=1e-6):
    lo, hi = np.percentile(x, [0.5, 99.5])
    x = np.clip(x, lo, hi)
    m, s = x.mean(), x.std()
    return (x - m) / (s + eps)

def volume_from_binary(mask_zyx, spacing):
    vox = float(spacing[0] * spacing[1] * spacing[2])
    return float(max(mask_zyx.sum() * vox, 0.0))

def volume_from_prob(prob_zyx, spacing):
    vox = float(spacing[0] * spacing[1] * spacing[2])
    return float(max(prob_zyx.sum() * vox, 0.0))

def volumetric_similarity(y_true, y_pred, eps=1e-4):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(1.0 - np.abs(y_true - y_pred) / (y_true + y_pred + eps)))

def parse_pid(path):
    return int(Path(path).name.split(".")[0])

def safe_percentile(arr, q):
    if arr.size == 0:
        return 0.0
    return float(np.percentile(arr, q))


In [ ]:
class AneurysmDataset(Dataset):
    def __init__(self, img_paths, mask_paths=None, augment=False):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.augment = augment

    def __len__(self):
        return len(self.img_paths)

    def _aug(self, x, y):
        if random.random() < 0.5:
            x = torch.flip(x, dims=[2]); y = torch.flip(y, dims=[2]) if y is not None else None
        if random.random() < 0.5:
            x = torch.flip(x, dims=[3]); y = torch.flip(y, dims=[3]) if y is not None else None
        if random.random() < 0.5:
            k = random.randint(0, 3)
            x = torch.rot90(x, k=k, dims=[2,3])
            if y is not None: y = torch.rot90(y, k=k, dims=[2,3])
        if random.random() < 0.7:
            x = x * (1.0 + random.uniform(-0.12, 0.12)) + random.uniform(-0.10, 0.10)
        if random.random() < 0.25:
            x = x + torch.randn_like(x) * 0.025
        return x, y

    def __getitem__(self, idx):
        p = self.img_paths[idx]
        img, spacing = load_nii(p)
        img = robust_zscore(img)
        img = np.transpose(img, (2,0,1)).astype(np.float32)
        img = torch.from_numpy(img).unsqueeze(0)
        out = {"image": img, "spacing": torch.tensor(spacing, dtype=torch.float32), "pid": parse_pid(p)}
        if self.mask_paths is not None:
            m, _ = load_nii(self.mask_paths[idx])
            m = (m > 0.5).astype(np.float32)
            m = np.transpose(m, (2,0,1)).astype(np.float32)
            m = torch.from_numpy(m).unsqueeze(0)
            if self.augment:
                img, m = self._aug(img, m)
                out["image"] = img
            out["mask"] = m
        return out


In [ ]:
def gn(ch):
    g = 8
    while ch % g != 0 and g > 1:
        g -= 1
    return nn.GroupNorm(g, ch)

class SE3D(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        hid = max(ch // r, 4)
        self.net = nn.Sequential(nn.AdaptiveAvgPool3d(1), nn.Conv3d(ch, hid, 1), nn.ReLU(inplace=True), nn.Conv3d(hid, ch, 1), nn.Sigmoid())
    def forward(self, x):
        return x * self.net(x)

class ResBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, drop=0.1):
        super().__init__()
        self.c1 = nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False)
        self.n1 = gn(out_ch)
        self.c2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.n2 = gn(out_ch)
        self.se = SE3D(out_ch)
        self.drop = nn.Dropout3d(drop)
        self.act = nn.ReLU(inplace=True)
        self.proj = nn.Conv3d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        i = self.proj(x)
        x = self.act(self.n1(self.c1(x)))
        x = self.n2(self.c2(x))
        x = self.se(x)
        x = self.drop(x)
        return self.act(x + i)

class AttnGate3D(nn.Module):
    def __init__(self, x_ch, g_ch, i_ch):
        super().__init__()
        self.wx = nn.Conv3d(x_ch, i_ch, 1, bias=False)
        self.wg = nn.Conv3d(g_ch, i_ch, 1, bias=False)
        self.psi = nn.Sequential(nn.ReLU(inplace=True), nn.Conv3d(i_ch, 1, 1), nn.Sigmoid())
    def forward(self, x, g):
        return x * self.psi(self.wx(x) + self.wg(g))

class AttentionUNet3D(nn.Module):
    def __init__(self, base=24):
        super().__init__()
        self.e1 = ResBlock3D(1, base)
        self.p1 = nn.MaxPool3d(2)
        self.e2 = ResBlock3D(base, base*2)
        self.p2 = nn.MaxPool3d(2)
        self.e3 = ResBlock3D(base*2, base*4)
        self.p3 = nn.MaxPool3d(2)
        self.b = ResBlock3D(base*4, base*8, drop=0.2)
        self.u3 = nn.ConvTranspose3d(base*8, base*4, 2, 2)
        self.a3 = AttnGate3D(base*4, base*4, base*2)
        self.d3 = ResBlock3D(base*8, base*4)
        self.u2 = nn.ConvTranspose3d(base*4, base*2, 2, 2)
        self.a2 = AttnGate3D(base*2, base*2, base)
        self.d2 = ResBlock3D(base*4, base*2)
        self.u1 = nn.ConvTranspose3d(base*2, base, 2, 2)
        self.a1 = AttnGate3D(base, base, max(base//2, 4))
        self.d1 = ResBlock3D(base*2, base)
        self.head = nn.Conv3d(base, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.p1(e1)); e3 = self.e3(self.p2(e2)); b = self.b(self.p3(e3))
        d3 = self.u3(b); d3 = self.d3(torch.cat([d3, self.a3(e3, d3)], dim=1))
        d2 = self.u2(d3); d2 = self.d2(torch.cat([d2, self.a2(e2, d2)], dim=1))
        d1 = self.u1(d2); d1 = self.d1(torch.cat([d1, self.a1(e1, d1)], dim=1))
        return self.head(d1)

class FocalTverskyBCELoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=1.33, bce_w=0.35, smooth=1e-5, pos_weight=4.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.bce_w = bce_w
        self.smooth = smooth
        self.pos_weight = pos_weight
    def forward(self, logits, target):
        pw = torch.tensor([self.pos_weight], device=logits.device, dtype=logits.dtype)
        bce = nn.functional.binary_cross_entropy_with_logits(logits, target, pos_weight=pw)
        p = torch.sigmoid(logits).reshape(logits.size(0), -1)
        t = target.reshape(target.size(0), -1)
        tp = (p*t).sum(dim=1)
        fp = (p*(1-t)).sum(dim=1)
        fn = ((1-p)*t).sum(dim=1)
        tv = (tp + self.smooth) / (tp + self.alpha*fn + self.beta*fp + self.smooth)
        ftl = ((1.0 - tv) ** self.gamma).mean()
        return self.bce_w * bce + (1.0 - self.bce_w) * ftl

class EMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
train_df["patient_id"] = train_df["patient_id"].astype(int)
all_train_imgs = sorted(TRAIN_IMG_DIR.glob("*.nii.gz"))
all_train_msks = [TRAIN_MASK_DIR / p.name for p in all_train_imgs]
all_test_imgs = sorted(TEST_IMG_DIR.glob("*.nii.gz"))
print("train:", len(all_train_imgs), "test:", len(all_test_imgs))


In [ ]:
CFG = {
    "n_splits": 5,
    "epochs": 72,
    "batch_size": 2,
    "num_workers": 0,
    "lr": 8e-4,
    "weight_decay": 1e-4,
    "base_ch": 24,
    "grad_clip": 1.0,
    "tta": True,
    "early_stop_patience": 14,
    "seeds": [42, 2025],
    "ema_decay": 0.995,
}
print(CFG)


In [ ]:
def make_splits(train_df, n_splits=5, seed=42):
    vols = train_df.sort_values("patient_id")["volume"].values
    try:
        bins = pd.qcut(vols, q=5, labels=False, duplicates="drop")
    except Exception:
        bins = pd.cut(vols, bins=5, labels=False)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(np.arange(len(all_train_imgs)), bins))

def infer_tta(model, x):
    model.eval(); out=[]
    with torch.no_grad():
        p = torch.sigmoid(model(x)); out.append(p)
        p = torch.sigmoid(model(torch.flip(x, dims=[3]))); out.append(torch.flip(p, dims=[3]))
        p = torch.sigmoid(model(torch.flip(x, dims=[4]))); out.append(torch.flip(p, dims=[4]))
        p = torch.sigmoid(model(torch.flip(x, dims=[3,4]))); out.append(torch.flip(p, dims=[3,4]))
    return torch.mean(torch.stack(out, 0), 0)

def postprocess_mask(prob, thr=0.5, min_vox=0, keep_largest=True, conf_pctl=95.0, conf_thr=0.0):
    m = (prob > thr).astype(np.uint8)
    if m.sum() == 0:
        return m
    lbl, n = ndimage.label(m)
    if n <= 1 and min_vox <= 0 and conf_thr <= 0.0:
        return m
    comp_ids = range(1, n+1)
    keep = []
    for cid in comp_ids:
        comp = (lbl == cid)
        sz = int(comp.sum())
        p95 = safe_percentile(prob[comp], conf_pctl)
        if sz >= int(min_vox) and p95 >= float(conf_thr):
            keep.append((cid, sz, p95))
    if len(keep) == 0:
        # fallback: keep largest component
        sizes = [(cid, int((lbl == cid).sum())) for cid in comp_ids]
        cid = sorted(sizes, key=lambda x: x[1], reverse=True)[0][0]
        out = (lbl == cid).astype(np.uint8)
        return out
    if keep_largest:
        cid = sorted(keep, key=lambda x: x[1], reverse=True)[0][0]
        return (lbl == cid).astype(np.uint8)
    out = np.zeros_like(m, dtype=np.uint8)
    for cid, _, _ in keep:
        out[lbl == cid] = 1
    return out

def train_fold(seed, fold, tr_idx, va_idx):
    seed_everything(seed + fold)
    tr_ds = AneurysmDataset([all_train_imgs[i] for i in tr_idx], [all_train_msks[i] for i in tr_idx], augment=True)
    va_ds = AneurysmDataset([all_train_imgs[i] for i in va_idx], [all_train_msks[i] for i in va_idx], augment=False)
    tr_ld = DataLoader(tr_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"])
    va_ld = DataLoader(va_ds, batch_size=1, shuffle=False, num_workers=CFG["num_workers"])

    model = AttentionUNet3D(base=CFG["base_ch"]).to(DEVICE)
    ema = EMA(model, decay=CFG["ema_decay"])
    crit = FocalTverskyBCELoss(alpha=0.7, beta=0.3, gamma=1.33, bce_w=0.35, pos_weight=4.0)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"], eta_min=5e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    sd = CKPT_DIR / f"seed_{seed}"
    sd.mkdir(parents=True, exist_ok=True)
    ck = sd / f"fold_{fold}.pt"
    best_vs, no_imp = -1.0, 0

    for ep in range(CFG["epochs"]):
        model.train(); losses=[]
        for b in tr_ld:
            x = b["image"].to(DEVICE)
            y = b["mask"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logit = model(x)
                loss = crit(logit, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(opt); scaler.update()
            ema.update(model)
            losses.append(loss.item())
        sch.step()

        # use EMA weights for validation
        eval_model = AttentionUNet3D(base=CFG["base_ch"]).to(DEVICE)
        ema.copy_to(eval_model)
        eval_model.eval()
        gts=[]; prs=[]
        with torch.no_grad():
            for b in va_ld:
                x = b["image"].to(DEVICE)
                m = b["mask"].cpu().numpy()[0,0]
                sp = b["spacing"].numpy()[0]
                p = torch.sigmoid(eval_model(x)).cpu().numpy()[0,0]
                pm = (p > 0.5).astype(np.uint8)
                gts.append(volume_from_binary(m, sp))
                prs.append(volume_from_binary(pm, sp))
        vs = volumetric_similarity(gts, prs)
        print(f"seed {seed} fold {fold} ep {ep+1:02d}/{CFG["epochs"]} loss={np.mean(losses):.4f} valVS={vs:.4f}")
        if vs > best_vs:
            best_vs = vs; no_imp = 0; torch.save(eval_model.state_dict(), ck)
        else:
            no_imp += 1
        if no_imp >= CFG["early_stop_patience"]:
            print(f"early stop seed {seed} fold {fold}, best={best_vs:.4f}")
            break
    return ck, best_vs

def collect_oof_probs(model_paths_by_seed, splits, model_weights_by_seed_fold):
    items=[]
    for fold, (_, va_idx) in enumerate(splits):
        models=[]
        ws=[]
        for seed, paths in model_paths_by_seed.items():
            m = AttentionUNet3D(base=CFG["base_ch"]).to(DEVICE)
            m.load_state_dict(torch.load(paths[fold], map_location=DEVICE))
            m.eval()
            models.append(m)
            ws.append(float(model_weights_by_seed_fold[(seed, fold)]))
        ws = np.asarray(ws, dtype=np.float64); ws = ws / (ws.sum() + 1e-12)

        va_ds = AneurysmDataset([all_train_imgs[i] for i in va_idx], [all_train_msks[i] for i in va_idx], augment=False)
        va_ld = DataLoader(va_ds, batch_size=1, shuffle=False)
        with torch.no_grad():
            for b in va_ld:
                pid = int(b["pid"][0]) if isinstance(b["pid"], torch.Tensor) else int(b["pid"])
                x = b["image"].to(DEVICE)
                sp = b["spacing"].numpy()[0].astype(np.float32)
                probs=[]
                for m in models:
                    p = infer_tta(m, x).cpu().numpy()[0,0] if CFG["tta"] else torch.sigmoid(m(x)).cpu().numpy()[0,0]
                    probs.append(p)
                probs = np.stack(probs, axis=0)
                prob = np.tensordot(ws, probs, axes=(0,0)).astype(np.float32)
                items.append({"pid": pid, "spacing": sp, "prob": prob})
    return items


In [ ]:
start = time.time()
splits = make_splits(train_df, n_splits=CFG["n_splits"], seed=42)
model_paths_by_seed = {}
model_weights_by_seed_fold = {}
rows=[]

for seed in CFG["seeds"]:
    seed_paths=[]
    for fold, (tr_idx, va_idx) in enumerate(splits):
        p, s = train_fold(seed, fold, tr_idx, va_idx)
        seed_paths.append(p)
        model_weights_by_seed_fold[(seed, fold)] = max(float(s), 1e-6)
        rows.append({"seed":seed, "fold":fold, "best_vs":float(s), "ckpt":str(p)})
    model_paths_by_seed[seed] = seed_paths

score_df = pd.DataFrame(rows)
score_df.to_csv(LOG_DIR / "fold_scores.csv", index=False)
print(score_df)
print(f"training done in {(time.time()-start)/60:.1f} min")


In [ ]:
# ===== OOF调参与校准 =====
oof_items = collect_oof_probs(model_paths_by_seed, splits, model_weights_by_seed_fold)
gt_map = {int(r.patient_id): float(r.volume) for _, r in train_df.iterrows()}

thr_grid = np.arange(0.32, 0.62, 0.01)
alpha_grid = [0.0, 0.25, 0.5, 0.75, 1.0]  # final = alpha*post_hard + (1-alpha)*soft
min_vox_grid = [0, 5, 10, 20]
keep_largest_grid = [True, False]
conf_thr_grid = [0.0, 0.35, 0.5]

best = {"vs": -1.0, "thr": 0.5, "alpha": 1.0, "min_vox": 0, "keep_largest": True, "conf_thr": 0.0}

for thr in thr_grid:
    for min_vox in min_vox_grid:
        for keep_largest in keep_largest_grid:
            for conf_thr in conf_thr_grid:
                hard=[]; soft=[]; y=[]
                for it in oof_items:
                    pid = int(it["pid"])
                    sp = it["spacing"]
                    prob = it["prob"]
                    m = postprocess_mask(prob, thr=thr, min_vox=min_vox, keep_largest=keep_largest, conf_pctl=95.0, conf_thr=conf_thr)
                    hard.append(volume_from_binary(m, sp))
                    soft.append(volume_from_prob(prob, sp))
                    y.append(gt_map[pid])
                hard = np.asarray(hard, dtype=np.float64)
                soft = np.asarray(soft, dtype=np.float64)
                y = np.asarray(y, dtype=np.float64)
                for a in alpha_grid:
                    pred = a * hard + (1.0 - a) * soft
                    vs = volumetric_similarity(y, pred)
                    if vs > best["vs"]:
                        best = {"vs": float(vs), "thr": float(thr), "alpha": float(a), "min_vox": int(min_vox), "keep_largest": bool(keep_largest), "conf_thr": float(conf_thr)}

print("Best OOF params:", best)

oof_rows=[]
for it in oof_items:
    pid = int(it["pid"])
    sp = it["spacing"]; prob = it["prob"]
    m = postprocess_mask(prob, thr=best["thr"], min_vox=best["min_vox"], keep_largest=best["keep_largest"], conf_pctl=95.0, conf_thr=best["conf_thr"])
    vh = volume_from_binary(m, sp)
    vs = volume_from_prob(prob, sp)
    blend = best["alpha"] * vh + (1.0 - best["alpha"]) * vs
    oof_rows.append({"patient_id": pid, "gt": gt_map[pid], "hard_pp": vh, "soft": vs, "blend": blend})

oof_df = pd.DataFrame(oof_rows).sort_values("patient_id").reset_index(drop=True)
k, b = np.polyfit(oof_df["blend"].values, oof_df["gt"].values, deg=1)
oof_df["calibrated"] = np.clip(k * oof_df["blend"].values + b, 0, None)
vs_uncal = volumetric_similarity(oof_df["gt"].values, oof_df["blend"].values)
vs_cal = volumetric_similarity(oof_df["gt"].values, oof_df["calibrated"].values)
use_cal = bool(vs_cal >= vs_uncal)
print(f"OOF VS uncal={vs_uncal:.6f} | cal={vs_cal:.6f} | use_cal={use_cal}")
oof_df.to_csv(PRED_DIR / "oof_predictions.csv", index=False)


In [ ]:
# ===== 测试推理 =====
all_models=[]
all_weights=[]
for seed, paths in model_paths_by_seed.items():
    for fold, p in enumerate(paths):
        m = AttentionUNet3D(base=CFG["base_ch"]).to(DEVICE)
        m.load_state_dict(torch.load(p, map_location=DEVICE))
        m.eval()
        all_models.append(m)
        all_weights.append(float(model_weights_by_seed_fold[(seed, fold)]))
all_weights = np.asarray(all_weights, dtype=np.float64)
all_weights = all_weights / (all_weights.sum() + 1e-12)

rows=[]
for p in tqdm(all_test_imgs, desc="Inference"):
    pid = parse_pid(p)
    arr, sp = load_nii(p)
    arr = robust_zscore(arr)
    arr = np.transpose(arr, (2,0,1)).astype(np.float32)
    x = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(DEVICE)
    probs=[]
    with torch.no_grad():
        for m in all_models:
            pr = infer_tta(m, x).cpu().numpy()[0,0] if CFG["tta"] else torch.sigmoid(m(x)).cpu().numpy()[0,0]
            probs.append(pr)
    probs = np.stack(probs, axis=0)
    prob = np.tensordot(all_weights, probs, axes=(0,0)).astype(np.float32)
    m = postprocess_mask(prob, thr=best["thr"], min_vox=best["min_vox"], keep_largest=best["keep_largest"], conf_pctl=95.0, conf_thr=best["conf_thr"])
    vh = volume_from_binary(m, sp)
    vs = volume_from_prob(prob, sp)
    blend = best["alpha"] * vh + (1.0 - best["alpha"]) * vs
    pred = np.clip(k * blend + b, 0, None) if use_cal else max(blend, 0.0)
    rows.append({"patient_id": pid, "volume": float(pred)})

sub = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)
sub.to_csv(PRED_DIR / "submission.csv", index=False)

targets = [
    Path("/root/setup/solution/working/submission.csv"),
    Path("/working/submission.csv"),
    OUTPUT_ROOT / "submission.csv",
]
saved=[]
for t in targets:
    try:
        t.parent.mkdir(parents=True, exist_ok=True)
        sub.to_csv(t, index=False)
        saved.append(str(t))
    except Exception as e:
        print("skip", t, e)

meta = {
    "run_name": RUN_NAME,
    "cfg": CFG,
    "best_oof": best,
    "calibration": {"use": use_cal, "k": float(k), "b": float(b), "vs_uncal": float(vs_uncal), "vs_cal": float(vs_cal)},
    "saved_submission_paths": saved,
}
with open(LOG_DIR / "run_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved submission paths:")
for s in saved:
    print(" -", s)
print(sub.head())


In [ ]:
# 提交前自检
for p in [Path("/root/setup/solution/working/submission.csv"), Path("/working/submission.csv"), PRED_DIR / "submission.csv"]:
    print(p, "exists=", p.exists())
    if p.exists():
        d = pd.read_csv(p)
        print(" shape=", d.shape, " cols=", d.columns.tolist())
        print(d.head(3))


## 参考方法（用于本版改进方向）
- nnU-Net: [Nature Methods 2021](https://www.nature.com/articles/s41592-020-01008-z), [GitHub](https://github.com/MIC-DKFZ/nnUNet)
- Attention U-Net: [arXiv:1804.03999](https://arxiv.org/abs/1804.03999)
- Tversky Loss: [arXiv:1706.05721](https://arxiv.org/abs/1706.05721)
- Focal Tversky: [arXiv:1810.07842](https://arxiv.org/abs/1810.07842)

这些方法在小目标、类别不平衡和医学分割挑战中常见且有效。
